# 🔺 Cuaderno 02 — Eliminación Gaussiana

**Métodos Numéricos · Unidad 3: Sistemas de Ecuaciones Lineales**

Este cuaderno resuelve $\mathbf{A}\mathbf{x}=\mathbf{b}$ con el método más
antiguo que sigue en uso: **eliminación gaussiana**. Al terminarlo sabrás
convertir un sistema en uno triangular, despejar de abajo hacia arriba, y
—lo más importante— **cuándo el método falla y por qué el pivoteo lo salva**.

### 🎯 Objetivos

- Convertir un sistema en **triangular superior** con operaciones de renglón.
- Programar la eliminación y la sustitución hacia atrás.
- Entender los **multiplicadores** $m_{ik}$ y por qué se dividen entre el pivote.
- Explicar con un ejemplo numérico **por qué el pivoteo parcial es necesario**.
- Calcular el **determinante** como producto de pivotes.
- Estimar el **costo** del método y saber cuándo conviene otro.

### 🧭 Cómo está armado

| § | Contenido |
|---|---|
| 1–4 | el método, paso a paso, con aritmética **exacta** |
| 5 | pivoteo: el pivote cero y el pivote diminuto |
| 6 | determinante y costo |
| 7 | ejercicios para completar |
| 8 | caso de ingeniería: circuito de 3 mallas |
| 9 | preguntas calificables |

> 💡 Todo el cuaderno trabaja con `M.sym[[...]]`, que guarda **enteros y
> fracciones exactas**. Así, cuando el resultado salga $1/2$, verás $1/2$ y no
> $0.4999999999999999$. Es la única forma honesta de estudiar un método directo:
> con flotantes, el error de redondeo se mezcla con el método y ya no sabes qué
> estás viendo.


In [ ]:
#@title ⚙️ Cargar entorno (ejecuta PRIMERO)

import os, sys, subprocess, importlib

try:
    import google.colab
    ES_COLAB = True
except ImportError:
    ES_COLAB = False

REPO_URL = "https://github.com/ITH-MGL-MN/U3_Sistemas_de_ecuaciones_lineales.git"


def _buscar_raiz(base, saltos=6):
    """Sube desde `base` hasta la carpeta que tiene config/config.yaml."""
    b = base
    for _ in range(saltos):
        if os.path.exists(os.path.join(b, "config", "config.yaml")):
            return os.path.abspath(b)
        padre = os.path.dirname(b)
        if padre == b:
            break
        b = padre
    return None


if ES_COLAB:
    # El repo se clona DENTRO de Drive, no en /content (que se borra al cerrar).
    from google.colab import drive
    drive.mount("/content/drive")

    BASE = "/content/drive/MyDrive/CursoMN"
    REPO = os.path.join(BASE, "U3_Repo")
    os.makedirs(BASE, exist_ok=True)
    try:
        if not os.path.exists(os.path.join(REPO, ".git")):
            print("Clonando el repositorio (la primera vez tarda un poco)...")
            subprocess.run(["git", "clone", REPO_URL, REPO], check=True)
        else:
            print("Actualizando el repositorio...")
            subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
        print("✅ Repositorio listo en", REPO)
    except Exception as _exc:
        print("⚠️ No pude clonar/actualizar el repositorio:", _exc)
else:
    _ruta_nb = globals().get("__vsc_ipynb_file__")
    REPO = _buscar_raiz(os.path.dirname(os.path.abspath(_ruta_nb))) if _ruta_nb else None
    REPO = REPO or _buscar_raiz(os.getcwd())

if REPO is None:
    raise RuntimeError(
        "No encuentro la carpeta del repositorio (la que tiene config/config.yaml).\n"
        "Abre el cuaderno desde dentro de U3_Repo o revisa que el clon de Colab exista."
    )

print("Repositorio :", REPO)

# En sys.path van las DOS raíces necesarias:
#   REPO/lib  -> robomat.py y el paquete mn_motor/
#   REPO      -> el paquete preguntas/
for _ruta in (os.path.join(REPO, "lib"), REPO):
    if _ruta not in sys.path:
        sys.path.insert(0, _ruta)

import robomat as _robomat
importlib.reload(_robomat)
from robomat import *               # noqa: F401,F403

# Explícitos, para que Pylance los vea y no dependamos de lo que reexporte
# la biblioteca.
import numpy as np
import sympy as sp

print("✅ Biblioteca:", _robomat.__file__)

# ============================================================
# Motor de calificación
# ============================================================
MODO_MOTOR = "modulos"      # "modulos" (depurar) | "blob" (lo que ve el alumno)
MODO = "blob" if ES_COLAB else MODO_MOTOR

if MODO == "modulos":
    try:
        for _n in [n for n in sys.modules
                   if n.split(".")[0] in ("mn_motor", "preguntas")]:
            del sys.modules[_n]
        import mn_motor as _motor
        from mn_motor import *      # noqa: F401,F403
    except ImportError as _exc:
        print("ℹ️ No pude importar el motor del repositorio:", _exc)
        MODO = "blob"

if MODO == "blob":
    import base64 as _b64
    import zlib as _zlib

    # Cada cuaderno tiene su propio blob: aquí va el del Cuaderno 02.
    ruta_motor = os.path.join(REPO, "grader", "u3_02_gaussiana.txt")
    with open(ruta_motor, encoding="utf-8") as _f:
        _blob = _f.read().strip()
    _codigo = _zlib.decompress(_b64.b64decode(_blob)).decode("utf-8")
    os.environ["U3_REPO"] = REPO
    _MOTOR = {"__name__": "blob_u3_02"}
    exec(_codigo, _MOTOR)
    globals().update({_n: _MOTOR[_n] for _n in _MOTOR["__all__"]})


# Este cuaderno califica SU conjunto de preguntas (carpeta preguntas/u3_02_gaussiana).
# En modo blob ya viene incrustado, pero en modo módulos hay que declararlo.
usar_cuaderno("u3_02_gaussiana", "U3_02_Eliminacion_Gaussiana")

print(f"✅ Motor cargado ({MODO}) · cuaderno {cuaderno()}")
print("   Sigue con la celda de identificación.")

## 0. Tu identificación

Cada alumno recibe **sus** matrices, sembradas con el Número de Control que sale
de tu correo. Copiar la respuesta del compañero da mal.


In [ ]:
#@title 👤 Identificación (correo institucional)
if ES_COLAB:
    import requests
    import google.auth
    import google.auth.transport.requests
    from google.colab import auth

    try:
        auth.authenticate_user()
        creds, _ = google.auth.default()
        creds.refresh(google.auth.transport.requests.Request())
        info = requests.get(
            "https://www.googleapis.com/drive/v3/about?fields=user",
            headers={"Authorization": "Bearer " + creds.token},
            timeout=20,
        ).json()
        cuenta = info.get("user", {})
        email = cuenta.get("emailAddress", "")
        nombre = cuenta.get("displayName", "")
    except Exception as e:
        print("⚠️ No pude obtener tu correo automáticamente.")
        print("   Vuelve a ejecutar y acepta el permiso en la ventana de Google.")
        print("   Detalle:", e)
        email, nombre = "", ""
else:
    email = "m16330887@hermosillo.tecnm.mx"      # <-- cambia por el tuyo
    nombre = ""

if email and email.lower().endswith(".tecnm.mx"):
    alumno_id = email
    print("✅ Identificado:", email)
    print("   Número de control:", extraer_nc(email))
else:
    from IPython.display import HTML, display as _display
    alumno_id = None
    _display(HTML(
        '<p style="color:#b00020;background:#ffe0e0;padding:12px;border-radius:8px;'
        'font-weight:bold">⛔ Necesito tu correo institucional (@….tecnm.mx).<br>'
        'Si iniciaste sesión con otra cuenta, ciérrala o usa una ventana de '
        'incógnita.</p>'))
    print("Correo detectado:", repr(email))

if alumno_id is None:
    raise RuntimeError("Detente: sin correo institucional no puedo generar tu tarea.")

MI_TAREA = generar_tarea(alumno_id, nombre)
print(f"\nListo: {len(MI_TAREA['preguntas'])} preguntas para el NC {MI_TAREA['nc']}.")


## 1. La idea: convertir el sistema en un triángulo

Un sistema $n\times n$ se escribe

$$\mathbf{A}\mathbf{x} = \mathbf{b}
\qquad\text{o, con la matriz aumentada,}\qquad
[\mathbf{A}\mid\mathbf{b}]$$

Resolverlo **tal cual** es incómodo: cada ecuación tiene todas las incógnitas
mezcladas. La eliminación gaussiana hace algo muy simple: **quitar incógnitas de
las ecuaciones de abajo**, hasta que la última tenga una sola.

$$\begin{bmatrix} a_{11} & a_{12} & a_{13} & \mid & b_1 \\
a_{21} & a_{22} & a_{23} & \mid & b_2 \\
a_{31} & a_{32} & a_{33} & \mid & b_3 \end{bmatrix}
\;\longrightarrow\;
\begin{bmatrix} u_{11} & u_{12} & u_{13} & \mid & d_1 \\
0 & u_{22} & u_{23} & \mid & d_2 \\
0 & 0 & u_{33} & \mid & d_3 \end{bmatrix}$$

La forma de la derecha se llama **triangular superior** y tiene una virtud: la
**última** ecuación ya está resuelta,

$$u_{33}x_3 = d_3 \;\Longrightarrow\; x_3 = \frac{d_3}{u_{33}}$$

Sabiendo $x_3$, la segunda ecuación tiene una sola incógnita, y así hacia arriba.
Eso es todo el método: **triangular primero, despejar después**.

### 1.1 El sistema que usaremos

$$\begin{aligned} 2x_1 + x_2 - x_3 &= 8 \\ -3x_1 - x_2 + 2x_3 &= -11 \\ -2x_1 + x_2 + 2x_3 &= -3 \end{aligned}$$

En forma matricial, con aritmética exacta:


In [ ]:
# M.sym guarda los números exactos: enteros y fracciones, no flotantes.
A = M.sym[[2, 1, -1],
          [-3, -1, 2],
          [-2, 1, 2]]
b = M.sym[[8], [-11], [-3]]
Ab = M.sym[[A, b]]        # matriz aumentada, al estilo de MATLAB

mostrar(Ab)
print("Forma:", Ab.shape, "| ¿exacta?", es_exacto(Ab))


### 1.2 Por qué no se puede simplemente "despejar"

Con dos ecuaciones se ve el problema:

$$\begin{bmatrix} 2 & 1 & \mid & 4 \\ 1 & 3 & \mid & 7 \end{bmatrix}$$

De la primera, $x_1 = (4 - x_2)/2$: depende de $x_2$. De la segunda,
$x_1 = 7 - 3x_2$: depende de $x_2$. **Ninguna se puede resolver sola.** Hay que
combinarlas para que una de las dos pierda una incógnita, y eso es exactamente
lo que hace la eliminación con operaciones de renglón.


## 2. Eliminación hacia adelante

Para borrar la incógnita $x_1$ del renglón $i$, se le resta al renglón $i$ un
múltiplo del renglón 1. El múltiplo se escoge para que el coeficiente de $x_1$
quede en cero:

$$R_i \leftarrow R_i - m_{ik}R_k,
\qquad\text{con}\qquad
m_{ik} = \frac{a_{ik}}{a_{kk}}$$

A $m_{ik}$ se le llama **multiplicador**. Es el número que hay que multiplicar
al pivote para igualar el coeficiente que queremos anular.

> ⚠️ Fíjate en el denominador: $a_{kk}$ es el **pivote**. Si vale cero, $m_{ik}$
> no existe y el método se detiene. Media unidad de este cuaderno (§5) trata de
> ese problema.

### 2.1 Paso $k=1$: eliminar $x_1$ de los renglones 2 y 3

Con $a_{11}=2$:

$$m_{21} = \frac{-3}{2}, \qquad m_{31} = \frac{-2}{2} = -1$$


In [ ]:
Ab1 = M.sym[[A, b]]          # copia limpia, para no alterar Ab

for i in (1, 2):            # renglones 2 y 3 (en Python: índices 1 y 2)
    pivote = Ab1[0, 0]
    m = Ab1[i, 0] / pivote
    print(f"m_{i+1},1 = {Ab1[i,0]} / {pivote} = {m}")
    Ab1[i, :] = Ab1[i, :] - m * Ab1[0, :]

print("\nDespués del paso k=1:")
mostrar(Ab1)
print("¿Sigue exacta?", es_exacto(Ab1))


Los dos ceros de la primera columna ya están. Ahora **el renglón 1 no se
vuelve a tocar**: la incógnita $x_1$ ya vive solo ahí.

### 2.2 Paso $k=2$: eliminar $x_2$ del renglón 3

El pivote ahora es $a_{22} = 1/2$, no el $a_{22}$ original: la eliminación del
paso anterior **modificó los coeficientes de abajo**. Por eso el pivote se lee
del estado actual de la matriz, nunca del sistema original.


In [ ]:
pivote = Ab1[1, 1]
m32 = Ab1[2, 1] / pivote
print(f"m_32 = {Ab1[2,1]} / {pivote} = {m32}")

Ab1[2, :] = Ab1[2, :] - m32 * Ab1[1, :]
print("\nDespués del paso k=2:")
mostrar(Ab1)


Ya está triangular. Comprueba que el patrón es el correcto: debajo de la
diagonal, puros ceros.

### 2.3 La eliminación, como función

El patrón de arriba se repite para cada columna. Encerrarlo en una función
permite reusarlo y, sobre todo, **probarlo**:


In [ ]:
def gauss_eliminar(A, b):
    """Devuelve la matriz aumentada triangularizada, SIN pivoteo.

    A y b deben ser exactas (M.sym) para que no haya error de redondeo.
    """
    Ab = M.sym[[A, b]]                      # copia: no toca los argumentos
    n = Ab.rows
    for k in range(n - 1):                  # columna del pivote
        for i in range(k + 1, n):           # renglones de abajo
            if Ab[k, k] == 0:
                raise ZeroDivisionError(
                    f"pivote cero en la columna {k+1}: hace falta pivoteo")
            m = Ab[i, k] / Ab[k, k]
            Ab[i, :] = Ab[i, :] - m * Ab[k, :]
    return Ab


mostrar(gauss_eliminar(A, b))


In [ ]:
# ¿Coincide con lo que hicimos a mano?
caso = gauss_eliminar(A, b)
print("¿Igual a mano?", caso == Ab1)


## 3. Sustitución hacia atrás

Con la matriz triangular, se despeja **de abajo hacia arriba**. Para la fila $i$:

$$x_i = \frac{d_i - \displaystyle\sum_{j=i+1}^{n} u_{ij}x_j}{u_{ii}}$$

La suma recorre solo las columnas **a la derecha** de la diagonal, que son las
incógnitas que ya conocemos. De ahí el nombre: se va de la última incógnita a la
primera.


In [ ]:
def sustitucion_atras(Ab):
    """Despeja x de una matriz aumentada triangular superior."""
    n = Ab.rows
    x = [None] * n
    for i in range(n - 1, -1, -1):          # de la última fila a la primera
        suma = sum(Ab[i, j] * x[j] for j in range(i + 1, n))
        if Ab[i, i] == 0:
            raise ZeroDivisionError(f"pivote cero en la fila {i+1}")
        x[i] = (Ab[i, n] - suma) / Ab[i, i]
    return M.sym[[[v] for v in x]]


x = sustitucion_atras(Ab1)
print("x =")
mostrar(x)


In [ ]:
# Contra la respuesta de la biblioteca: deben ser idénticas.
esperado = A.solve(b)
print("nuestra x  :", x.T.tolist())
print("A.solve(b) :", esperado.T.tolist())
print("¿Idénticas?:", x == esperado)

# Y el residual tiene que ser exactamente cero: A@x - b = 0
mostrar(A @ x - b)
print("¿Residual exactamente cero?", (A @ x - b).is_zero_matrix)


## 4. El método completo

Juntando las dos piezas queda el resolvedor. Nótese la **verificación** que se
hace al final: resolver no es solo obtener números, es comprobar que cumplen el
sistema original. Es un hábito que conviene adoptar desde ahora.


In [ ]:
def gauss_solve(A, b):
    """Resuelve A x = b por eliminación gaussiana (sin pivoteo). Devuelve x."""
    Ab = gauss_eliminar(A, b)
    x = sustitucion_atras(Ab)
    # Comprobación: si el residual no es cero, algo salió mal.
    if not (A @ x - b).is_zero_matrix:
        raise ArithmeticError("el residual no es cero: revisa la eliminación")
    return x


x = gauss_solve(A, b)
mostrar(x)
print("¿Exacta toda la cuenta?", es_exacto(x))


### 4.1 ¿Y si el sistema no tiene solución única?

La eliminación **no necesita** saber si hay solución para avanzar: si aparece un
pivote cero, es que $\mathbf{A}$ no es invertible. O el sistema no tiene
solución, o tiene infinitas. El método se detiene; distinguir cuál de los dos
casos es, es tema de otro curso (rango). Aquí basta saber que **un pivote cero
es una señal**, no un accidente.


## 5. Pivoteo: cuando el método se rompe

### 5.1 El pivote que vale cero

$$\begin{bmatrix} 0 & 1 & \mid & 1 \\ 1 & 1 & \mid & 3 \end{bmatrix}$$

El sistema tiene solución única ($x_1=2$, $x_2=1$), pero $m_{21} = 1/0$ **no
existe**. El método se cae aunque el problema esté perfectamente bien planteado.

La solución es de sentido común: si el renglón de arriba empieza con cero y el de
abajo no, **intercámbialos**. Eso es el **pivoteo parcial**.


In [ ]:
P = M.sym[[0, 1], [1, 1]]
pb = M.sym[[1], [3]]
print("det =", P.det(), "(≠ 0, así que SÍ hay solución única)")

try:
    gauss_solve(P, pb)
except ZeroDivisionError as exc:
    print("⛔", exc)

# Con un intercambio de renglones se arregla. Ojo: P[1,:] es una MATRIZ (1x2),
    # y M.sym[[matriz, matriz]] las concatena. Para apilar renglones hacen
    # falta listas, así que se pasan con .tolist().
    P2 = M.sym[[P.tolist()[1], P.tolist()[0]]]
    b2 = M.sym[[[pb[1, 0]], [pb[0, 0]]]]
print("\ntras intercambiar renglones:")
mostrar(M.sym[[P2, b2]])
print("x =", gauss_solve(P2, b2).T.tolist())


### 5.2 El pivote diminuto: el caso que de verdad asusta

Un pivote cero al menos **avisa**. El peligroso es el pivote *diminuto*, porque
no avisa: el método corre hasta el final y devuelve una basura.

$$\begin{bmatrix} 10^{-18} & 1 & \mid & 1 \\ 1 & 1 & \mid & 2 \end{bmatrix}$$

La solución exacta es $x_1 = 1, x_2 = 1$. Veamos qué pasa con aritmética
exacta y qué pasa con flotantes:


In [ ]:
eps = sp.Rational(1, 10**18)      # sp viene de robomat (sympy reexportado)
Ae = M.sym[[eps, 1], [1, 1]]
be = M.sym[[1], [2]]

print("EXACTO  ->", gauss_solve(Ae, be).T.tolist())

# Lo mismo, pero con flotantes (como lo haría NumPy)
Ae_f = np.array([[1e-18, 1.0], [1.0, 1.0]])
be_f = np.array([1.0, 2.0])
print("FLOAT   ->", np.linalg.solve(Ae_f, be_f).round(6).tolist())


Con flotantes, $x_1$ sale $\approx 111$ en vez de $1$. El error no viene de
"pocos dígitos": viene de **dividir entre un pivote casi cero**, que amplifica
cualquier error de redondeo por un factor de $10^{18}$.

Arreglo: elegir como pivote el **mayor en valor absoluto** de la columna. Así el
multiplicador nunca pasa de 1 en magnitud, y el error no se amplifica.

$$|m_{ik}| = \left|\frac{a_{ik}}{a_{kk}}\right| \le 1$$

Eso es el **pivoteo parcial**. Con él, el ejemplo anterior da la respuesta
correcta:


In [ ]:
def gauss_pivoteo(A, b):
    """Igual que gauss_solve, pero elige el pivote mayor de cada columna.

    Versión NUMÉRICA (float): aquí el pivoteo tiene sentido porque los números
    se pueden comparar por magnitud. Con símbolos no siempre se puede.
    """
    Ab = np.array(M[[A, b]], dtype=float) if not isinstance(A, np.ndarray) else \
         np.hstack([np.asarray(A, float), np.asarray(b, float).reshape(-1, 1)])
    n = Ab.shape[0]

    for k in range(n - 1):
        p = k + np.argmax(np.abs(Ab[k:, k]))     # renglón con el pivote mayor
        if p != k:
            Ab[[k, p], :] = Ab[[p, k], :]        # intercambio de renglones
        for i in range(k + 1, n):
            m = Ab[i, k] / Ab[k, k]
            Ab[i, :] = Ab[i, :] - m * Ab[k, :]

    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (Ab[i, -1] - Ab[i, i+1:n] @ x[i+1:n]) / Ab[i, i]
    return x


print("FLOAT con pivoteo ->", gauss_pivoteo(Ae_f, be_f).round(6).tolist())
print("             (la respuesta correcta es [1.0, 1.0])")


### 5.3 La consecuencia para este cuaderno

Con aritmética **exacta** no hay error de redondeo que amplificar, así que el
pivote diminuto no molesta: el pivote cero sí, porque la división es imposible.
Por eso el material hace las dos cosas:

| Versión | Para qué | Qué enseña |
|---|---|---|
| **exacta** (`M.sym`, sin pivoteo) | verificar el algoritmo | el método es correcto |
| **flotante** (NumPy, con pivoteo) | resolver de verdad | el redondeo existe y el pivoteo lo controla |

Un curso que solo hiciera la primera te dejaría creyendo que los flotantes son
inofensivos. Uno que solo hiciera la segunda te dejaría sin saber si el error que
ves es del método o del redondeo.


## 6. Determinante y costo

### 6.1 El determinante sale casi gratis

Las operaciones de renglón del tipo $R_i \leftarrow R_i - mR_k$ **no cambian** el
determinante. Los intercambios sí: cada uno lo multiplica por $-1$. Como la
matriz triangular tiene el mismo determinante que su diagonal, queda

$$\det(\mathbf{A}) = (-1)^{p}\prod_{k=1}^{n} u_{kk}$$

con $p$ el número de intercambios. O sea: ya que triangulizaste, el determinante
es multiplicar la diagonal.


In [ ]:
Ab_tri = gauss_eliminar(A, b)
producto = sp.prod([Ab_tri[k, k] for k in range(Ab_tri.rows)])
print("producto de los pivotes:", producto)
print("A.det()                :", A.det())
print("¿Iguales?", producto == A.det())

# Con el circuito del caso de ingeniería (más grande)
print("\nCaso del circuito:", R.det() if 'R' in dir() else "(se define en la §8)")


### 6.2 Cuánto cuesta

Contar operaciones importa porque es lo que decide si un método sirve para un
sistema de $1000\times1000$ o solo para uno de juguete. Para eliminación
gaussiana:

$$C_{\text{elim}} = \sum_{k=1}^{n-1}(n-k)\left[2(n-k)+3\right]
= \frac{2}{3}n^{3} + \frac{1}{2}n^{2} - \frac{7}{6}n$$

$$C_{\text{sust}} = n^{2}$$

El término dominante es $\frac{2}{3}n^{3}$: **el costo crece con el cubo de
$n$**. Duplicar el tamaño del sistema multiplica el trabajo por ocho.


In [ ]:
print(f"{'n':>6} {'eliminación':>14} {'sustitución':>13} {'total':>14}")
for n in (3, 10, 100, 1000):
    e = sp.Rational(2,3)*n**3 + sp.Rational(1,2)*n**2 - sp.Rational(7,6)*n
    s = n**2
    print(f"{n:>6} {str(e):>14} {s:>13} {str(e+s):>14}")
print("\nDe n=100 a n=1000 el trabajo se multiplica por ~1000 (el cubo).")


## 7. Tu turno

Completa las funciones. Las celdas de abajo las prueban solas: si algo está mal,
te dicen **qué** está mal, no solo que está mal.


In [ ]:
# ✏️ EJERCICIO 1 — completa la eliminación hacia adelante
def mi_eliminacion(A, b):
    """Devuelve la matriz aumentada triangular superior, sin pivoteo.

    Pista: para cada columna k, y para cada renglón i debajo del pivote,
    calcula m = Ab[i,k] / Ab[k,k] y resta m veces el renglón k.
    """
    Ab = M.sym[[A, b]]
    n = Ab.rows
    # TODO: escribe aquí los dos ciclos
    return None


resultado = mi_eliminacion(A, b)
print("resultado:", "todavía no" if resultado is None else "listo")


In [ ]:
# ✅ Verificación del ejercicio 1 (no modificar)
if resultado is None:
    print("⏳ El ejercicio 1 sigue pendiente.")
else:
    if verificar(resultado, Ab1):
        print("✅ Ejercicio 1 correcto")
        print("   Además es exacto:", es_exacto(resultado))
    else:
        print("❌ Ejercicio 1 incorrecto")
        print("   Se esperaba:")
        mostrar(Ab1)
        if hasattr(resultado, "shape") and resultado.shape != Ab1.shape:
            print(f"   Tu resultado tiene forma {resultado.shape}, se esperaba {Ab1.shape}")
        else:
            print("   Revisa los índices: el pivote de la columna k está en Ab[k,k],")
            print("   y solo se modifican los renglones i > k.")


In [ ]:
# ✏️ EJERCICIO 2 — completa la sustitución hacia atrás
def mi_sustitucion(Ab):
    """Devuelve x como columna, a partir de la aumentada triangular."""
    n = Ab.rows
    x = [None] * n
    # TODO: recorre las filas de abajo hacia arriba y despeja
    return None


mi_x = mi_sustitucion(mi_eliminacion(A, b)) if resultado is not None else None
print("x:", "todavía no" if mi_x is None else mi_x.T.tolist())


In [ ]:
# ✅ Verificación del ejercicio 2 (no modificar)
if mi_x is None:
    print("⏳ El ejercicio 2 sigue pendiente.")
elif verificar(mi_x, A.solve(b)):
    print("✅ Ejercicio 2 correcto:", mi_x.T.tolist())
    print("   Residual A@x - b:", (A @ M.sym[[mi_x[0,0]],[mi_x[1,0]],[mi_x[2,0]]] - b).T.tolist()
          if hasattr(mi_x, "rows") else "—")
else:
    print("❌ Ejercicio 2 incorrecto")
    print("   Se esperaba x =", A.solve(b).T.tolist())
    print("   Recuerda: la suma va sobre las columnas a la DERECHA de la diagonal.")


In [ ]:
# ✏️ EJERCICIO 3 — sin escribir ni un ciclo
#
# Resuelve este sistema con SOLO lo que ya sabes de la biblioteca,
# usando aritmética exacta, y guarda el resultado en `x_facil`.

C3 = M.sym[[4, -2,  1],
           [-2, 4, -2],
           [1, -2,  4]]
d3 = M.sym[[11], [-16], [17]]

x_facil = None      # TODO: una sola expresión
print("x_facil:", "todavía no" if x_facil is None else x_facil)


In [ ]:
# ✅ Verificación del ejercicio 3 (no modificar)
if x_facil is None:
    print("⏳ El ejercicio 3 sigue pendiente.")
else:
    x3 = x_facil if hasattr(x_facil, "rows") else var(x_facil)
    if verificar(x3, C3.solve(d3)):
        print("✅ Ejercicio 3 correcto:", x3.T.tolist())
        print("   ¿Exacto?", es_exacto(x3), "· ¿Residual cero?", (C3 @ x3 - d3).is_zero_matrix)
    else:
        print("❌ Ejercicio 3 incorrecto. Se esperaba:", C3.solve(d3).T.tolist())
        print("   Pista: la biblioteca ya sabe resolver sistemas; solo hay que pedírselo.")


## 8. Caso de ingeniería: un circuito de tres mallas

Un circuito resistivo con tres mallas produce, por la ley de voltajes de
Kirchhoff, un sistema lineal cuya matriz es **simétrica** y con la diagonal
dominante:

$$\begin{bmatrix} R_3+R_1 & -R_1 & 0 \\ -R_1 & R_1+R_2+R_4 & -R_2 \\
0 & -R_2 & R_2+R_5 \end{bmatrix}
\begin{bmatrix} i_1 \\ i_2 \\ i_3 \end{bmatrix}
= \begin{bmatrix} V_1 \\ V_2 \\ V_3 \end{bmatrix}$$

| Resistencia | Valor | Dónde está |
|---|---|---|
| $R_1$ | $2\ \Omega$ | compartida entre mallas 1 y 2 |
| $R_2$ | $4\ \Omega$ | compartida entre mallas 2 y 3 |
| $R_3$ | $6\ \Omega$ | solo malla 1 |
| $R_4$ | $10\ \Omega$ | solo malla 2 |
| $R_5$ | $8\ \Omega$ | solo malla 3 |

Con esas resistencias la matriz queda numérica y pequeña, así que se puede
resolver **exactamente**:


In [ ]:
R1_, R2_, R3_, R4_, R5_ = 2, 4, 6, 10, 8

R = M.sym[[R3_ + R1_, -R1_,               0],
          [-R1_,     R1_ + R2_ + R4_, -R2_],
          [0,       -R2_,              R2_ + R5_]]
V = M.sym[[20], [22], [4]]

print("Matriz del circuito:")
mostrar(R)
print("\n¿Simétrica?", R == R.T, "| ¿det?", R.det())
print("Dominancia diagonal por fila:")
for i in range(3):
    propia = abs(R[i, i])
    otras = sum(abs(R[i, j]) for j in range(3) if j != i)
    print(f"  fila {i+1}: |diagonal| = {propia} vs suma de las otras = {otras}"
          f"  {'✓' if propia > otras else '✗'}")


In [ ]:
# Resolver con el método del cuaderno
i_mallas = gauss_solve(R, V)
print("Corrientes de malla (A):")
mostrar(i_mallas)

# Comprobar contra la biblioteca y contra el residual
print("A.solve(V) :", R.solve(V).T.tolist())
print("¿Coinciden?:", i_mallas == R.solve(V))
print("\nResidual R@i - V:")
mostrar(R @ i_mallas - V)
print("¿Cero exacto?", (R @ i_mallas - V).is_zero_matrix)


In [ ]:
# ¿Cuánto costó? El costo predicho por la fórmula, para n = 3
n = 3
predicho = sp.Rational(2,3)*n**3 + sp.Rational(1,2)*n**2 - sp.Rational(7,6)*n + n**2
print("operaciones predichas para n=3:", predicho)

# Y qué habría pasado con pivoteo (numérico), para comparar
print("con gauss_pivoteo (float):", gauss_pivoteo(R, V).round(12).tolist())
print("exacto                   :", [float(v) for v in i_mallas])


### 8.1 ¿Gauss es el método correcto para este caso?

Sí, y conviene saber **por qué**:

| Criterio | Este caso | Consecuencia |
|---|---|---|
| Tamaño | $n = 3$ | el costo $O(n^3)$ es irrelevante |
| Estructura | densa (casi todos los coeficientes ≠ 0) | no hay atajo que explotar |
| Simetría | sí, $\mathbf{R} = \mathbf{R}^\top$ | se podría usar Cholesky, pero no hace falta |
| Diagonal dominante | sí | converge también por métodos iterativos, pero con $n=3$ no vale la pena |
| Necesitas $\det$ o $i$ | solo las corrientes | Gauss resuelve y ya |

**Cuándo NO usarías Gauss aquí:**

- Si el circuito tuviera **50 mallas en cadena** (matriz tridiagonal), el
  algoritmo de **Thomas** cuesta $O(n)$ en vez de $O(n^3)$: para $n=50$ es unas
  17 veces menos trabajo. Eso se ve en el Cuaderno 04.
- Si el circuito tuviera **miles de mallas** y hubiera que resolverlo para
  muchos vectores $\mathbf{V}$ distintos, conviene **descomposición LU**
  (Cuaderno 03): se paga una vez la factorización y cada $\mathbf{V}$ nuevo
  cuesta solo $O(n^2)$.
- Si la matriz fuera **dispersa y muy grande** (miles de nodos), los
  **iterativos** ganan porque no destruyen los ceros.

La regla corta: **Gauss es el método por omisión para un sistema denso y
chico**. Los otros existen para explotar estructura o para reusar trabajo.


In [ ]:
# Comprobación física: la potencia total disipada debe ser positiva
# P = sum(i_k^2 * R_propia_k)  con las resistencias propias de cada malla
propias = [R3_ + R1_, R1_ + R2_ + R4_, R2_ + R5_]
P = sum(float(i_mallas[k])**2 * propias[k] for k in range(3))
print("Potencia disipada (W):", round(P, 4), "| ¿positiva?", P > 0)

# Y los voltajes de malla que pediría el problema inverso
print("\nSi las corrientes fueran [3,2,1] A, las fuentes serían V =",
      (R @ M.sym[[3],[2],[1]]).T.tolist())


## 9. Preguntas calificables

Ocho preguntas sobre este cuaderno. Cinco de teoría y tres donde escribes el
resultado de **tus** matrices.

### Cómo se contesta

1. Ejecuta la celda de un bloque de preguntas.
2. Elige la opción, o escribe la respuesta (`[[1,2,3],[4,5,6]]` para una matriz).
3. Pulsa **Guardar respuesta**. Sin ese botón no se guarda, aunque ya la veas escrita.
4. Cuando termines, ejecuta **Calificar** (no envía nada).
5. Si ya estás conforme, ejecuta **Enviar**.

| | |
|---|---|
| 👀 Calificar sin enviar | las veces que quieras |
| 📤 Envíos al profesor | **5** como máximo |


In [ ]:
#@title 🧪 Preguntas 1 a 4 · el método y los multiplicadores
pregunta(1)
pregunta(2)
pregunta(3)
pregunta(4)


In [ ]:
#@title 🧪 Preguntas 5 a 8 · con tus matrices
pregunta(5)
pregunta(6)
pregunta(7)
pregunta(8)


In [ ]:
#@title 📝 Calificar (sin enviar) — úsala las veces que quieras
calificar()


In [ ]:
#@title 📤 Enviar la calificación (máximo 5 intentos)
enviar(alumno_id)


In [ ]:
#@title 🔎 Comprobar lo que ya está guardado (no envía nada)
consultar_calificacion()


---
### ✅ ¿Qué sigue?

Ya resuelves sistemas con el método directo más general. En el **Cuaderno 03**
verás cómo **reusar** la eliminación: una vez factorizada $\mathbf{A} =
\mathbf{L}\mathbf{U}$, resolver para un $\mathbf{b}$ nuevo cuesta mucho menos
que empezar de cero.

📖 La teoría completa, en [`docs/metodos/01_eliminacion_gaussiana.md`](https://github.com/ITH-MGL-MN/U3_Sistemas_de_ecuaciones_lineales/blob/main/docs/metodos/01_eliminacion_gaussiana.md).
